# FDP2 — EDGAR API: Menarik Fakta Tanpa API Key
**Tujuan notebook ini:** mempraktikkan alur FDP2 secara real — memetakan ticker → CIK, memanggil EDGAR companyfacts API dengan header User-Agent yang benar, lalu mengekstrak deret laba bersih tahunan Microsoft (MSFT) langsung dari SEC.

> Dijalankan: 4 Nov 2026. Jika Anda menjalankan ulang, angka bisa bertambah (filing baru) — itu fitur, bukan bug.

In [1]:
# Tujuan cell ini: siapkan library dan header User-Agent (wajib untuk EDGAR API)
import json
import urllib.request
from datetime import date

HEADERS = {"User-Agent": "Septian FDP-Learn septian@example.com"}  # SEC menolak request tanpa User-Agent (HTTP 403)
print("headers siap:", HEADERS)

headers siap: {'User-Agent': 'Septian FDP-Learn septian@example.com'}


## Penjelasan (Indonesia)
EDGAR API **gratis tanpa API key**, tapi punya 2 aturan keras: (1) header `User-Agent` harus berisi nama + email — tanpanya Anda dapat HTTP 403; (2) maksimal 10 request/detik. Ini "error #1" pemula EDGAR: lupa User-Agent lalu mengira API-nya mati.

In [1]:
# Tujuan cell ini: petakan ticker MSFT menjadi CIK (Central Index Key)
url = "https://www.sec.gov/files/company_tickers.json"
req = urllib.request.Request(url, headers=HEADERS)
tickers = json.load(urllib.request.urlopen(req, timeout=30))
cik = next(v["cik_str"] for v in tickers.values() if v["ticker"] == "MSFT")
print(f"MSFT -> CIK {cik} (zero-pad 10 digit: {cik:010d})")

MSFT -> CIK 789019 (zero-pad 10 digit: 0000789019)


## Penjelasan (Indonesia)
CIK adalah ID permanen emiten di SEC (tidak seperti ticker yang bisa berubah). File `company_tickers.json` adalah kamus ticker → CIK resmi. Setelah dapat CIK, kita panggil endpoint `companyfacts`: `https://data.sec.gov/api/xbrl/companyfacts/CIK{{cik10}}.json` — satu JSON raksasa berisi SEMUA fakta XBRL emiten tersebut.

In [1]:
# Tujuan cell ini: unduh companyfacts MSFT dan intip strukturnya
cik10 = f"{cik:010d}"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
req = urllib.request.Request(url, headers=HEADERS)
facts = json.load(urllib.request.urlopen(req, timeout=60))
print("emiten:", facts["entityName"])
ugaap = facts["facts"]["us-gaap"]
print("concept tersedia:", len(ugaap))
ni = ugaap["NetIncomeLoss"]["units"]["USD"]
print("fakta NetIncomeLoss (USD):", len(ni))
print("contoh satu fakta:")
print(json.dumps(ni[0], indent=1)[:500])

emiten: MICROSOFT CORP
concept tersedia: 562
fakta NetIncomeLoss (USD): 347
contoh satu fakta:
{
 "start": "2007-07-01",
 "end": "2007-09-30",
 "val": 4289000000,
 "accn": "0001193125-10-171791",
 "fy": 2010,
 "fp": "FY",
 "form": "10-K",
 "filed": "2010-07-30",
 "frame": "CY2007Q3"
}


## Penjelasan (Indonesia)
Struktur JSON: `facts → taksonomi (us-gaap) → concept → units (USD) → list fakta`. Tiap fakta punya `end` (akhir periode), `val` (nilai), `accn` (ID filing), `fy/fp` (tahun/kuartal fiskal), `form` (10-K/10-Q), `filed` (tanggal lapor), dan untuk *flow* ada `start`. Fakta `instant` (neraca) tidak punya `start` — persis seperti pelajaran M38: flow vs stock adalah struktur data, bukan cuma teori.

> Quirk nyata: concept `Revenues` untuk MSFT ternyata jarang di snapshot ini (emiten kadang memakai tag turunan). Pelajaran FDP4: jangan hardcode satu concept — siapkan fallback.

In [1]:
# Tujuan cell ini: ekstrak deret laba bersih tahunan (aturan FDP4: filed terbaru menang)
def is_annual(f):
    if f.get("form") != "10-K" or f.get("fp") != "FY":
        return False
    if "start" in f:  # duration (flow)
        return (date.fromisoformat(f["end"]) - date.fromisoformat(f["start"])).days > 300
    return True  # instant (stock)

facts_ni = ugaap["NetIncomeLoss"]["units"]["USD"]
annual = [f for f in facts_ni if is_annual(f)]

best = {}  # end -> fakta dengan filed terbaru (aturan FDP4)
for f in annual:
    e = f["end"]
    if e not in best or f["filed"] > best[e]["filed"]:
        best[e] = f

series = sorted(best.items())
print(f"{'Tahun':<8} {'Periode':<12} {'Laba bersih (USD mld)':<22} {'Filed':<12} {'Provenance (accn)'}")
for end, f in series[-6:]:
    print(f"{f['fy']:<8} {end:<12} {f['val']/1e9:<22.1f} {f['filed']:<12} {f['accn']}")

Tahun    Periode      Laba bersih (USD mld)  Filed        Provenance (accn)
2023     2021-06-30   61.3                   2023-07-27   0000950170-23-035122
2024     2022-06-30   72.7                   2024-07-30   0000950170-24-087843
2025     2023-06-30   72.4                   2025-07-30   0000950170-25-100235
2026     2024-06-30   88.1                   2026-07-29   0001193125-26-323660
2026     2025-06-30   101.8                  2026-07-29   0001193125-26-323660
2026     2026-06-30   133.7                  2026-07-29   0001193125-26-323660


## Penjelasan (Indonesia)
Deret di atas adalah output FDP2 + FDP4 dalam 20 baris: fetch → filter fakta tahunan → deduplikasi via `filed` terbaru. Perhatikan kolom provenance (`accn`): tiap angka bisa dilacak ke filing persisnya. Inilah yang dimaksud "angka tanpa provenance = rumor yang diformat rapi".

**Tugas mandiri:** ganti `"MSFT"` menjadi ticker lain (mis. `"AAPL"`, `"NVDA"`), jalankan ulang, dan bandingkan deretnya. Itulah gunanya pipeline: ganti satu string, dapat satu emiten baru.